# Quickstart: classify a point cloud

This first notebook takes you from a fresh install to a class prediction on a 3D point cloud, and introduces the three ideas the rest of the library is built on:

- the **model registry** and the `create_model` factory (the `timm`-style entry point),
- the **packed-batch** tensor format (the PyTorch Geometric convention used everywhere here),
- how a model maps points to logits, and how to read the prediction back.

Every cell runs on CPU in a few seconds. Follow-ups: [segment a scene](02-segmentation-inference.md), [preprocessing pipelines](03-transforms.md), [your own data](04-custom-dataset.md), and [training](05-training.md).

In [ ]:
# On Colab, install the library first (uncomment):
# !pip install "torch-pointcloud[pyg-lib]"

import torch

import torch_pointcloud as tp

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch-pointcloud", tp.__version__, "| torch", torch.__version__, "| device:", device)

## 1. Find a model in the registry

Models are built by name through a single factory, `create_model`, mirroring `timm.create_model`. Names follow the pattern `<arch>-<variant>.<dataset>`, for example `pointnet2-ssg.modelnet40.xu-yan`.

List what is available for a task with `list_models` (it accepts a glob):

In [ ]:
from torch_pointcloud.models import list_models

list_models("pointnet2*", task="classification")

## 2. Build a model

`create_model(name, task=...)` returns a ready `nn.Module`. Two flags matter:

- `pretrained=True` loads the registered weights (cached locally). We leave it off here so the cell runs anywhere with no download: you get the architecture with random weights.
- `return_info=True` also returns the registry entry, including the exact transform pipeline the checkpoint was trained with (used in the [segmentation notebook](02-segmentation-inference.md)).

The factory returns a typed `ClassificationModel`, so `.num_classes`, `.eval()`, and `.to(device)` work as usual.

In [ ]:
model = tp.create_model("pointnet2-ssg.modelnet40.xu-yan", task="classification").eval().to(device)

n_params = sum(p.numel() for p in model.parameters())
print(type(model).__name__, "| classes:", model.num_classes, "| parameters:", f"{n_params:,}")

## 3. The packed-batch format

A batch of point clouds is stored *packed*: all points concatenated into one flat tensor, with a companion `batch` vector giving each point's cloud index. Nothing is padded to a common size.

| tensor  | shape    | meaning                                                  |
| ------- | -------- | -------------------------------------------------------- |
| `pos`   | $(N, 3)$ | XYZ of every point in the batch, concatenated            |
| `x`     | $(N, C)$ | optional per-point features (`None` uses coordinates only) |
| `batch` | $(N,)$   | cloud index in $[0, B)$ for each point                   |

For $B$ clouds with $N_i$ points each, $N = N_1 + \cdots + N_B$. A reduction over one cloud becomes a `scatter` op on `batch`, never a Python loop. Let us build a batch of two toy spheres:

In [ ]:
def sphere(n: int) -> torch.Tensor:
    v = torch.randn(n, 3)
    return v / v.norm(dim=1, keepdim=True)


cloud_a, cloud_b = sphere(2048), sphere(1536)

pos = torch.cat([cloud_a, cloud_b], dim=0)
batch = torch.cat([
    torch.zeros(len(cloud_a), dtype=torch.long),
    torch.ones(len(cloud_b), dtype=torch.long),
])

print("pos:", tuple(pos.shape), "| batch:", tuple(batch.shape), "| clouds:", int(batch.max()) + 1)

Building `batch` by hand is fine for a demo; the `collate` helper does it for you (see [Use your own data](04-custom-dataset.md)). Here is a quick look at the first cloud, with a small helper we reuse across the notebooks:

In [ ]:
import matplotlib.pyplot as plt


def show_cloud(pos, color=None, *, ax=None, title=None, size=6, cmap="viridis"):
    """Scatter a point cloud. `pos` is (N, 3); `color` is per-point RGB, a label vector, or None."""
    if ax is None:
        ax = plt.figure(figsize=(4, 4)).add_subplot(projection="3d")

    p = pos.detach().cpu().numpy()
    c = color.detach().cpu().numpy() if torch.is_tensor(color) else color
    kw = {} if c is None else {"cmap": cmap}
    ax.scatter(p[:, 0], p[:, 1], p[:, 2], c=c, s=size, depthshade=False, linewidths=0, **kw)
    ax.set_box_aspect((1, 1, 1))
    ax.set_axis_off()
    if title:
        ax.set_title(title, fontsize=10)
    return ax

show_cloud(cloud_a, title=f"toy sphere - {len(cloud_a)} points");

## 4. Run the model

A classification model is called as `model(x, pos, batch)`: features first (here `None`), then coordinates, then the batch index. It returns one logit vector per cloud, shape $(B, \text{num\_classes})$.

In [ ]:
with torch.no_grad():
    logits = model(None, pos.to(device), batch.to(device))

print("logits:", tuple(logits.shape))  # (2, 40): one row per cloud

## 5. Read the prediction

Turn logits into probabilities with a softmax, take the top-k, and map indices to ModelNet40 class names.

> Our weights are random, so the labels below are meaningless: they show the mechanics. Load `pretrained=True` for a real prediction.

In [ ]:
from torch_pointcloud.datasets.modelnet import MODELNET40_CLASSES

probs = logits.softmax(dim=-1)
topk = probs.topk(3, dim=-1)
for i in range(probs.shape[0]):
    names = (MODELNET40_CLASSES[j] for j in topk.indices[i].tolist())
    scores = (f"{s:.2f}" for s in topk.values[i].tolist())
    print(f"cloud {i}: " + ", ".join(f"{n} ({s})" for n, s in zip(names, scores)))

## Using real pretrained weights

The only change for a real prediction is `pretrained=True`:

```python
model = tp.create_model(
    "pointnet2-ssg.modelnet40.xu-yan", 
    task="classification", 
    pretrained=True,
).eval()
```

The weights download to a local cache on first use. Reproducing a checkpoint's reported accuracy also means matching the preprocessing it was trained with, and `create_model(..., return_info=True)` hands you that exact transform pipeline.

## Next steps

- [Segment a scene](02-segmentation-inference.md): dense per-point labels and the inferer contract.
- [Preprocessing pipelines](03-transforms.md): compose transforms step by step.
- [Use your own data](04-custom-dataset.md): build a `Dataset` and collate it into packed batches.
- [Train a model](05-training.md): an end-to-end loop with PyTorch Lightning.